In this Notebook, we prepare the data from the UN General Debate Corpus, from *Understanding State Preferences With Text As Data: Introducing the UN General Debate Corpus* (https://journals.sagepub.com/doi/epub/10.1177/2053168017712821)

The next cell contains all relevant parameters to be modified.

In [4]:
"""
Parameters

Set both load variables to False if this is the first time running the notebook. 
"""
load_highspace_vectors = True
load_reduced_vectors = True

# Where do you want to store the data?
data_path = "./input-data" 
parquet_file_name = "UNGDC-demo-30pc-dataset.pq"
embedding_file_name = "UNGDC-embedding-vectors.npy"
umap_file_name = "UNGDC-reduced-vectors.npy"

# The data will be chunked to approximately this length:
agglomerate_chunk_length = 250

# If the computation is too slow, cut down the dataset:
# We keep the top `keep_percentage`% chunks, ordered by mutual information
filter_by_information = True
keep_percentage = 30

# Embedding & reduction params
model_name = 'all-mpnet-base-v2' # for sentence transformers

umap_params = {
    'n_components': 2
}


First load the data from CSV and take a look at it. The column `text` contains the transcript text. These transcripts are quite long on average, so we will chunk them.


In [7]:
import pandas as pd
import numpy as np
# Import the CSV
df = pd.read_csv('input-data/un-general-debates.csv')

length = df['text'].apply(lambda x: len(x))
print(f"Average transcript length: {length.mean():.0f} characters")
df.head()

Average transcript length: 17967 characters


,Unnamed: 0,session,year,country,text
0,0,44,1989,MDV,﻿It is indeed a pleasure for me and the member...
1,1,44,1989,FIN,"﻿\nMay I begin by congratulating you. Sir, on ..."
2,2,44,1989,NER,"﻿\nMr. President, it is a particular pleasure ..."
3,3,44,1989,URY,﻿\nDuring the debate at the fortieth session o...
4,4,44,1989,ZWE,﻿I should like at the outset to express my del...


The transcripts are broken into paragraphs by newlines, so we will keep it simple and chunk on newlines. However, this gives fairly short chunks, so we will combine chunks together in sequence up to a maximum size of `agglomerate_chunk_length`.

In [ ]:
# Reload the dataset as otherwise this cell is not idempotent.
df = pd.read_csv('un-general-debates.csv')

def chunk_speech(text, agglomerate_chunk_length):
    chunks = text.split('\n')
    chunks = [s for s in chunks if len(s)<500]
    chunks = merge_strings_until_N(chunks, agglomerate_chunk_length)
    return chunks

def merge_strings_until_N(strings_list, N, separator="\n"):
    merged_results = []
    current_merge = []
    current_length = 0
    sep_len = len(separator)

    for string in strings_list:
        string_len = len(string)
        
        # Calculate potential new length, including the separator if it's not the first item
        potential_length = current_length + sep_len + string_len if current_length > 0 else string_len
        
        if potential_length <= N:
            current_merge.append(string)
            current_length = potential_length
        else:
            if current_merge:
                merged_results.append(separator.join(current_merge))
            current_merge = [string]
            current_length = string_len
            
    if current_merge:
        merged_results.append(separator.join(current_merge))

    return merged_results

df['chunk'] = df['text'].apply(lambda x: chunk_speech(x, agglomerate_chunk_length))
# Reformat the chunks into their own documents
df = df.explode('chunk')

print(f"There are now {len(df)} chunks")

Now that we've chunked the dataset, there is a lot of data. When we do our topic modelling, this can be slow and generate more cluster names then we can effectively visualize. For that reason, I chose to filter the dataset by taking the top `keep_percentage`% of chunks, weighted by information.
(See: https://vectorizers.readthedocs.io/en/latest/information_weight_transform.html)

In [ ]:
if filter_by_information:
    from sklearn.feature_extraction.text import CountVectorizer, TfidfTransformer
    from vectorizers.transformers  import InformationWeightTransformer
    
    # Step 1: Convert text to term frequency counts
    count_vectorizer = CountVectorizer(max_features=1000)
    count_matrix = count_vectorizer.fit_transform(df['chunk'])
    
    # Step 2: Apply TF-IDF transformation to the count matrix
    #fidf_transformer = TfidfTransformer()
    iw_transformer = InformationWeightTransformer()
    iw_matrix = iw_transformer.fit_transform(count_matrix)
    
    # Sum TF-IDF scores per chunk (informativeness score)
    df['information_weight'] = iw_matrix.sum(axis=1).A1
    
    # Keep top `keep_percentage`%
    threshold = df['information_weight'].quantile(1-keep_percentage/100)
    df_filtered = df[df['information_weight'] >= threshold]
    print(f"There are now {len(df_filtered)} chunks")
    df = df_filtered
    df.head()

Next, we do the standard topic modelling pipeline of embedding the documents as high dimensional vectors and then using UMAP to reduce that to 2 dimensions

In [ ]:
from tqdm import tqdm
from pathlib import Path
# Load or Generate High-Space Vectors
if load_highspace_vectors:
    vector_folder_path = Path(data_path+"/vectors")
    embeddings = np.load(str(vector_folder_path)+'/'+embedding_file_name, allow_pickle=True)
    df['embedding'] = list(embeddings)
    embedding_vectors = df['embedding'].to_numpy()
    print(f"Loaded embedding vectors from {str(vector_folder_path)+'/'+embedding_file_name}")
else:
    from sentence_transformers import SentenceTransformer
    import torch
    # Check if GPU is available
    device = 'cuda' if torch.cuda.is_available() else 'cpu'
    print(f"Using device: {device}")
    
    # Load the embedding model
    # Options: 'all-MiniLM-L6-v2' (fast, 384 dim), 'all-mpnet-base-v2' (better quality, 768 dim)
    model = SentenceTransformer(model_name, device=device)
    print(f"Loaded model: {model_name}")
    print(f"Embedding dimension: {model.get_sentence_embedding_dimension()}")
    
    # Assuming df has a 'text' column with the debate speeches
    # Adjust the column name if needed
    text_column = 'chunk'  # Change this to your actual column name
    
    # Get the texts from your dataframe
    texts = df[text_column].tolist()
    
    # Generate embeddings
    # batch_size can be adjusted based on your GPU memory
    batch_size = 32
    print(f"Embedding {len(texts)} documents...")
    
    embeddings = model.encode(
        texts,
        batch_size=batch_size,
        show_progress_bar=True,
        convert_to_numpy=True,
        normalize_embeddings=True  # L2 normalization for cosine similarity
    )
    df['embedding'] = list(embeddings)
    
    embedding_vectors = df['embedding'].to_numpy()
    vector_folder_path = Path(data_path+"/vectors")
    vector_folder_path.mkdir(parents=True, exist_ok=True)
    np.save(str(vector_folder_path)+'/embedding_vectors.npy', embedding_vectors)
    print("Embedding complete!")
    print(f"Each document is now represented as a {embeddings.shape[1]}-dimensional vector")


embedding_vectors = np.vstack(embedding_vectors)

In [ ]:
# Load or Generate reduced vectors
if load_reduced_vectors:
    reduced_embeddings = np.load(str(vector_folder_path)+'/'+umap_file_name, allow_pickle=True)
    df['reduced'] = list(reduced_embeddings)
    reduced_vectors = df['reduced'].to_numpy()
    print(f"Loaded reduced vectors from {str(vector_folder_path)+'/'+umap_file_name}")
else:
    import umap  
    reducer = umap.UMAP(**umap_params)
    reduced_embeddings = reducer.fit_transform(embedding_vectors)
    print(f"UMAP reduction complete. Shape: {reduced_embeddings.shape}")
    df['reduced'] = list(reduced_embeddings)
    reduced_vectors = df['reduced'].to_numpy()
    np.save(str(vector_folder_path)+'/'+umap_file_name, reduced_vectors)

reduced_vectors = np.vstack(reduced_vectors)
text = df['chunk'].to_numpy()
time = df['year'].to_numpy()

In [ ]:
## Finally sort everything by time

sorted_idx = np.argsort(time)
time = time[sorted_idx]
text = text[sorted_idx]
reduced_vectors = reduced_vectors[sorted_idx]
embedding_vectors = embedding_vectors[sorted_idx]

df.to_parquet(data_path+"/"+parquet_file_name)
df.head()